In [1]:
import xarray as xr
import numpy as np

# Official PS 26066 region
LON_MIN, LON_MAX = 45, 105
LAT_MIN, LAT_MAX = 5, 30
TARGET_RES = 0.25
STANDARD_DEPTHS = [0, 5, 10, 20, 30, 50, 75, 100, 125, 150, 200, 300, 500, 700, 1000]

In [2]:
import copernicusmarine

glorys_temp = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao"],
    minimum_longitude=LON_MIN, maximum_longitude=LON_MAX,
    minimum_latitude=LAT_MIN, maximum_latitude=LAT_MAX,
    start_datetime="2023-01-01", end_datetime="2023-01-31",  # start with one month to test
    minimum_depth=0, maximum_depth=1000,
)
print(glorys_temp)

c:\Users\LENOVO\miniconda3\envs\oceanembed\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO - 2026-09-11T15:39:08Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register
INFO - 2026-09-11T15:39:23Z - Selected dataset version: "202311"
INFO - 2026-09-11T15:39:23Z - Selected dataset part: "default"
WARNING - 2026-09-11T15:39:23Z - Some of your subset selection [0.0, 1000.0] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]


<xarray.Dataset> Size: 2GB
Dimensions:    (depth: 35, latitude: 301, longitude: 721, time: 31)
Coordinates:
  * depth      (depth) float32 140B 0.494 1.541 2.646 ... 643.6 763.3 902.3
  * latitude   (latitude) float32 1kB 5.0 5.083 5.167 5.25 ... 29.83 29.92 30.0
  * longitude  (longitude) float32 3kB 45.0 45.08 45.17 ... 104.8 104.9 105.0
  * time       (time) datetime64[ns] 248B 2023-01-01 2023-01-02 ... 2023-01-31
Data variables:
    thetao     (time, depth, latitude, longitude) float64 2GB dask.array<chunksize=(2, 25, 301, 721), meta=np.ndarray>
Attributes: (12/25)
    Conventions:               CF-1.4
    bulletin_date:             2021-07-07 00:00:00
    bulletin_type:             operational
    comment:                   CMEMS product
    domain_name:               GL12
    easting:                   longitude
    ...                        ...
    references:                http://www.mercator-ocean.fr
    source:                    MERCATOR GLORYS12V1
    title:              

In [ ]:
new_lat = np.arange(LAT_MIN, LAT_MAX, TARGET_RES)
new_lon = np.arange(LON_MIN, LON_MAX, TARGET_RES)

glorys_regridded = glorys_temp.interp(latitude=new_lat, longitude=new_lon, method="linear")
glorys_standard_depths = glorys_regridded.interp(depth=STANDARD_DEPTHS, method="linear")
print(glorys_standard_depths['thetao'].shape)

(31, 15, 100, 240)


In [4]:
import requests
import warnings
from urllib3.exceptions import InsecureRequestWarning
warnings.filterwarnings("ignore", category=InsecureRequestWarning)

incois_url = (
    "https://erddap.incois.gov.in/erddap/griddap/incois_argo_mnt_VAM.nc?"
    "TEMP[(2023-01-15T00:00:00Z)][(5):(1000)][(5):(30)][(45):(105)]"
)
response = requests.get(incois_url, timeout=60, verify=False)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

with open("data/raw/incois_argo_test.nc", "wb") as f:
    f.write(response.content)

import xarray as xr
argo_gridded = xr.open_dataset("data/raw/incois_argo_test.nc")
print(argo_gridded)

Status code: 200
Content-Type: application/x-netcdf
<xarray.Dataset> Size: 117kB
Dimensions:    (time: 1, ZAX: 19, latitude: 25, longitude: 61)
Coordinates:
  * time       (time) datetime64[ns] 8B 2023-01-15
  * ZAX        (ZAX) float64 152B 5.0 10.0 20.0 30.0 ... 700.0 800.0 900.0 1e+03
  * latitude   (latitude) float64 200B 5.5 6.5 7.5 8.5 ... 26.5 27.5 28.5 29.5
  * longitude  (longitude) float64 488B 45.5 46.5 47.5 ... 103.5 104.5 105.5
Data variables:
    TEMP       (time, ZAX, latitude, longitude) float32 116kB ...
Attributes: (12/29)
    CDI:                        Climate Data Interface version 2.5.4 (https:/...
    cdm_data_type:              Grid
    CDO:                        Climate Data Operators version 2.5.4 (https:/...
    Conventions:                CF-1.6, COARDS, ACDD-1.3
    Easternmost_Easting:        105.5
    frequency:                  mon
    ...                         ...
    standard_name_vocabulary:   CF Standard Name Table v29
    summary:                

In [5]:
argo_gridded = argo_gridded.rename({"ZAX": "depth"})
print(argo_gridded['depth'].values)

[   5.   10.   20.   30.   50.   75.  100.  125.  150.  200.  250.  300.
  400.  500.  600.  700.  800.  900. 1000.]


In [6]:
import copernicusmarine
results = copernicusmarine.describe(contains=["ssh"])

Fetching catalogue 1: 100%|██████████| 2/2 [00:49<00:00, 24.94s/it]


In [7]:
for product in results.products:
    for dataset in product.datasets:
        if "ssh" in dataset.dataset_id.lower():
            print(dataset.dataset_id)

cmems_mod_bal_phy-ssh_anfc_detided_P1D-m
cmems_mod_blk_phy-ssh_anfc_2.5km_P1D-m
cmems_mod_blk_phy-ssh_anfc_2.5km_P1M-m
cmems_mod_blk_phy-ssh_anfc_2.5km_PT15M-i
cmems_mod_blk_phy-ssh_anfc_2.5km_PT1H-m
cmems_mod_blk_phy-ssh_anfc_detided-2.5km_P1D-m
cmems_mod_blk_phy-ssh_anfc_mrm-500m_P1D-m
cmems_mod_blk_phy-ssh_anfc_mrm-500m_PT1H-i
cmems_mod_blk_phy-ssh_my_2.5km-climatology_P1M-m
cmems_mod_blk_phy-ssh_my_2.5km_P1D-m
cmems_mod_blk_phy-ssh_my_2.5km_P1M-m
cmems_mod_blk_phy-ssh_my_2.5km_P1Y-m
cmems_mod_ibi_phy-ssh_anfc_detided-0.027deg_P1D-m
cmems_mod_ibi_phy-ssh_anfc_detided-0.027deg_P1M-m
cmems_mod_ibi_phy-ssh_my_0.027deg_P1D-m
cmems_mod_ibi_phy-ssh_my_0.027deg_P1M-m
cmems_mod_ibi_phy-ssh_my_0.027deg_P1Y-m
cmems_mod_ibi_phy-ssh_my_0.027deg_PT1H-m
cmems_mod_ibi_phy-ssh_my_detided-0.027deg_P1D-m
cmems_mod_ibi_phy-ssh_my_detided-0.027deg_P1M-m
cmems_mod_ibi_phy-ssh_my_detided-0.027deg_P1Y-m
cmems_obs-ins_eur_phy-ssh_my_tide-surge_PT1H-m
cmems_obs-ins_glo_phy-ssh_my_na_PT1H
cmems_obs-ins_glo_p

In [8]:
ssh = copernicusmarine.open_dataset(
    dataset_id="cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D",
    variables=["sla"],
    minimum_longitude=LON_MIN, maximum_longitude=LON_MAX,
    minimum_latitude=LAT_MIN, maximum_latitude=LAT_MAX,
    start_datetime="2023-01-01", end_datetime="2023-01-31",
)
print(ssh)

INFO - 2026-09-11T15:40:33Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register
INFO - 2026-09-11T15:40:54Z - Selected dataset version: "202411"
INFO - 2026-09-11T15:40:54Z - Selected dataset part: "default"


<xarray.Dataset> Size: 24MB
Dimensions:    (time: 31, latitude: 200, longitude: 480)
Coordinates:
  * latitude   (latitude) float32 800B 5.062 5.188 5.312 ... 29.69 29.81 29.94
  * longitude  (longitude) float32 2kB 45.06 45.19 45.31 ... 104.7 104.8 104.9
  * time       (time) datetime64[ns] 248B 2023-01-01 2023-01-02 ... 2023-01-31
Data variables:
    sla        (time, latitude, longitude) float64 24MB dask.array<chunksize=(10, 200, 480), meta=np.ndarray>
Attributes: (12/43)
    Conventions:                     CF-1.6
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    cdm_data_type:                   Grid
    comment:                         Sea Surface Height measured by Altimetry...
    contact:                         servicedesk.cmems@mercator-ocean.eu
    creator_email:                   servicedesk.cmems@mercator-ocean.eu
    ...                              ...
    time_coverage_duration:          P1D
    time_coverage_end:               2023-12-31T12:00:00

In [9]:
from argopy import DataFetcher
test = DataFetcher().region([80, 100, 5, 20, 0, 100, '2023-01-01', '2023-01-10']).to_xarray()
print(test)

<xarray.Dataset> Size: 39kB
Dimensions:          (N_POINTS: 328)
Coordinates:
    LATITUDE         (N_POINTS) float64 3kB 5.806 5.806 5.806 ... 14.96 14.96
    LONGITUDE        (N_POINTS) float64 3kB 85.95 85.95 85.95 ... 87.62 87.62
    TIME             (N_POINTS) datetime64[ns] 3kB 2023-01-01T03:23:36 ... 20...
  * N_POINTS         (N_POINTS) int64 3kB 0 1 2 3 4 5 ... 323 324 325 326 327
Data variables: (12/15)
    CYCLE_NUMBER     (N_POINTS) int64 3kB 108 108 108 108 ... 109 109 109 109
    DATA_MODE        (N_POINTS) <U1 1kB 'D' 'D' 'D' 'D' 'D' ... 'D' 'D' 'D' 'D'
    DIRECTION        (N_POINTS) <U1 1kB 'A' 'A' 'A' 'A' 'A' ... 'A' 'A' 'A' 'A'
    PLATFORM_NUMBER  (N_POINTS) int64 3kB 2902769 2902769 ... 2902766 2902766
    POSITION_QC      (N_POINTS) int64 3kB 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1
    PRES             (N_POINTS) float32 1kB 0.3 1.0 2.0 2.9 ... 74.7 85.3 95.3
    ...               ...
    PSAL_ERROR       (N_POINTS) float32 1kB 0.01 0.01 0.01 ... 0.01 0.01 0.01
    

In [10]:
import earthaccess
earthaccess.login()  # prompts for your Earthdata username/password once, then caches it

results_currents = earthaccess.search_data(
    short_name="OSCAR_L4_OC_FINAL_V2.0",
    temporal=("2023-01-01", "2023-01-31"),
    bounding_box=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
)
print(f"Found {len(results_currents)} files")
files_currents = earthaccess.download(results_currents, "data/raw/currents")

c:\Users\LENOVO\miniconda3\envs\oceanembed\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
c:\Users\LENOVO\miniconda3\envs\oceanembed\Lib\site-packages\earthaccess\store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


Found 31 files


QUEUEING TASKS | : 100%|██████████| 31/31 [00:00<00:00, 2391.19it/s]
PROCESSING TASKS | : 100%|██████████| 31/31 [00:00<00:00, 57507.04it/s]
COLLECTING RESULTS | : 100%|██████████| 31/31 [00:00<?, ?it/s]


In [11]:
results_winds = earthaccess.search_data(
    short_name="CCMP_WINDS_10M6HR_L4_V3.1",
    temporal=("2023-01-01", "2023-01-31"),
    bounding_box=(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
)
print(f"Found {len(results_winds)} files")
files_winds = earthaccess.download(results_winds, "data/raw/winds")

Found 31 files


QUEUEING TASKS | : 100%|██████████| 31/31 [00:00<00:00, 3802.19it/s]
PROCESSING TASKS | : 100%|██████████| 31/31 [00:00<?, ?it/s]
COLLECTING RESULTS | : 100%|██████████| 31/31 [00:00<?, ?it/s]


In [12]:
import xarray as xr

winds_ds = xr.open_mfdataset("data/raw/winds/*.nc")
print(winds_ds)

currents_ds = xr.open_mfdataset("data/raw/currents/*.nc")
print(currents_ds)

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 124, latitude: 720, longitude: 1440)
Coordinates:
  * latitude   (latitude) float32 3kB -89.88 -89.62 -89.38 ... 89.38 89.62 89.88
  * longitude  (longitude) float32 6kB 0.125 0.375 0.625 ... 359.4 359.6 359.9
  * time       (time) datetime64[ns] 992B 2023-01-01 ... 2023-01-31T18:00:00
Data variables:
    uwnd       (time, latitude, longitude) float32 514MB dask.array<chunksize=(1, 720, 1440), meta=np.ndarray>
    vwnd       (time, latitude, longitude) float32 514MB dask.array<chunksize=(1, 720, 1440), meta=np.ndarray>
    ws         (time, latitude, longitude) float32 514MB dask.array<chunksize=(1, 720, 1440), meta=np.ndarray>
    nobs       (time, latitude, longitude) float32 514MB dask.array<chunksize=(1, 720, 1440), meta=np.ndarray>
Attributes: (12/54)
    contact:                       Remote Sensing Systems, support@remss.com
    Conventions:                   CF-1.7 ACDD-1.3
    data_structure:                grid
    title:      

In [13]:
winds_regional = winds_ds.sel(
    latitude=slice(LAT_MIN, LAT_MAX),
    longitude=slice(LON_MIN, LON_MAX)
)

In [14]:
import numpy as np

# lat/lon are 1D dask arrays despite showing as (latitude,)/(longitude,) — compute them once
currents_ds = currents_ds.assign_coords(
    latitude=("latitude", currents_ds["lat"].values),
    longitude=("longitude", currents_ds["lon"].values),
)
currents_regional = currents_ds.sel(
    latitude=slice(LAT_MIN, LAT_MAX),
    longitude=slice(LON_MIN, LON_MAX)
)
print(currents_regional['u'].shape)

(31, 241, 101)


In [15]:
winds_daily = winds_regional.resample(time="1D").mean()
print(winds_daily)

<xarray.Dataset> Size: 12MB
Dimensions:    (time: 31, latitude: 100, longitude: 240)
Coordinates:
  * latitude   (latitude) float32 400B 5.125 5.375 5.625 ... 29.38 29.62 29.88
  * longitude  (longitude) float32 960B 45.12 45.38 45.62 ... 104.4 104.6 104.9
  * time       (time) datetime64[ns] 248B 2023-01-01 2023-01-02 ... 2023-01-31
Data variables:
    uwnd       (time, latitude, longitude) float32 3MB dask.array<chunksize=(1, 100, 240), meta=np.ndarray>
    vwnd       (time, latitude, longitude) float32 3MB dask.array<chunksize=(1, 100, 240), meta=np.ndarray>
    ws         (time, latitude, longitude) float32 3MB dask.array<chunksize=(1, 100, 240), meta=np.ndarray>
    nobs       (time, latitude, longitude) float32 3MB dask.array<chunksize=(1, 100, 240), meta=np.ndarray>
Attributes: (12/54)
    contact:                       Remote Sensing Systems, support@remss.com
    Conventions:                   CF-1.7 ACDD-1.3
    data_structure:                grid
    title:                  

In [18]:
import numpy as np

# Your official grid, defined once
common_lat = np.arange(LAT_MIN, LAT_MAX, 0.25)
common_lon = np.arange(LON_MIN, LON_MAX, 0.25)

def regrid(ds, var):
    return ds[var].interp(latitude=common_lat, longitude=common_lon, method="linear")

sst_grid    = regrid(ds_sst, "thetao").isel(depth=0)   # surface only
sss_grid    = regrid(sss, "sos").isel(depth=0)
ssh_grid    = ssh["sla"].interp(latitude=common_lat, longitude=common_lon, method="linear")
uwnd_grid   = winds_daily["uwnd"].interp(latitude=common_lat, longitude=common_lon, method="linear")
vwnd_grid   = winds_daily["vwnd"].interp(latitude=common_lat, longitude=common_lon, method="linear")
ucurr_grid  = currents_regional["u"].interp(latitude=common_lat, longitude=common_lon, method="linear")
vcurr_grid  = currents_regional["v"].interp(latitude=common_lat, longitude=common_lon, method="linear")

print(sst_grid.shape, ssh_grid.shape, uwnd_grid.shape, ucurr_grid.shape)

NameError: name 'ds_sst' is not defined